<a href="https://colab.research.google.com/github/trang1981/ELAPS/blob/main/SantanderABC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Dán toàn bộ vào MỘT ô Colab. Tạo nhánh A/B trên CÙNG khách hàng và nhãn.
# Sửa từ code placebo Santander: không lấy mẫu pseudo-cutoff, không lặp 5 seed.
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = Path('/content/drive/MyDrive')
INPUT_FILE = None  # Có thể gán Path('/content/drive/MyDrive/.../Santander_CreditCard.csv')
OUTPUT_PARENT = ROOT / 'Santander_B_same_population'
SPLIT_SEED = 42

LAST_VALUE_FEATURES = [
    'ind_empleado', 'pais_residencia', 'sexo', 'age', 'ind_nuevo',
    'antiguedad', 'indrel', 'indrel_1mes', 'tiprel_1mes', 'indresi',
    'indext', 'conyuemp', 'indfall', 'tipodom', 'cod_prov', 'nomprov',
    'ind_actividad_cliente', 'renta', 'segmento',
]
FIRST_VALUE_FEATURES = ['canal_entrada']
MODEL_FEATURES = LAST_VALUE_FEATURES + FIRST_VALUE_FEATURES
assert len(MODEL_FEATURES) == 20

if INPUT_FILE is None:
    found = sorted(ROOT.rglob('Santander_CreditCard.csv'))
    if len(found) != 1:
        raise FileNotFoundError(
            f'Cần đúng một Santander_CreditCard.csv; tìm thấy {len(found)}: '
            f'{[str(p) for p in found[:10]]}. Hãy đặt INPUT_FILE cụ thể.'
        )
    INPUT_FILE = found[0]
if not INPUT_FILE.is_file():
    raise FileNotFoundError(INPUT_FILE)

run_dir = OUTPUT_PARENT / datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%S%fZ')
run_dir.mkdir(parents=True, exist_ok=False)
print('Đọc:', INPUT_FILE)
print('Lưu:', run_dir)

cols = ['ncodpers', 'fecha_dato', 'fecha_alta', 'ind_tjcr_fin_ult1'] + MODEL_FEATURES
df = pd.read_csv(INPUT_FILE, usecols=cols, low_memory=False)
df['fecha_dato'] = pd.to_datetime(df['fecha_dato'], errors='coerce')
df['fecha_alta'] = pd.to_datetime(df['fecha_alta'], errors='coerce')
df['ind_tjcr_fin_ult1'] = pd.to_numeric(df['ind_tjcr_fin_ult1'], errors='coerce')
df = df.dropna(subset=['ncodpers', 'fecha_dato', 'fecha_alta', 'ind_tjcr_fin_ult1']).copy()
if not df['ind_tjcr_fin_ult1'].isin([0, 1]).all():
    raise ValueError('Nhãn sở hữu thẻ phải là 0/1 sau khi đọc dữ liệu.')

# Giống code gốc: chỉ xét các snapshot trong năm quan hệ đầu tiên.
df['end_observation'] = df['fecha_alta'] + pd.DateOffset(years=1)
df = df.loc[(df['fecha_dato'] >= df['fecha_alta']) &
            (df['fecha_dato'] < df['end_observation'])].copy()
df = df.sort_values(['ncodpers', 'fecha_dato'], kind='mergesort')
duplicates = int(df.duplicated(['ncodpers', 'fecha_dato']).sum())
if duplicates:
    raise ValueError(f'Có {duplicates} snapshot trùng khách hàng-ngày; cần quy tắc xử lý trước.')

# Landmark Santander là snapshot thứ HAI của mỗi khách hàng, không phải ngày thứ 60.
df['snapshot_no'] = df.groupby('ncodpers', sort=False).cumcount() + 1
event = (df.loc[df['ind_tjcr_fin_ult1'].eq(1)]
         .groupby('ncodpers')['fecha_dato'].min().rename('event_date'))
meta = (df.groupby('ncodpers', sort=False)
        .agg(fecha_alta=('fecha_alta', 'first'),
             first_snapshot=('fecha_dato', 'min'),
             last_snapshot=('fecha_dato', 'max'),
             n_snapshots=('fecha_dato', 'size'),
             end_observation=('end_observation', 'first'))
        .join(event))
second = df.loc[df['snapshot_no'].eq(2)].set_index('ncodpers')['fecha_dato']
meta = meta.join(second.rename('landmark_date'))

# Cùng eligibility và outcome cho A/B: đã có 2 snapshot trước quyết định,
# chưa sở hữu thẻ tại mốc 2, có ít nhất một snapshot sau mốc để theo dõi.
eligible = (meta['n_snapshots'].ge(3) & meta['landmark_date'].notna() &
            (meta['last_snapshot'] > meta['landmark_date']) &
            (meta['event_date'].isna() | (meta['event_date'] > meta['landmark_date'])))
meta = meta.loc[eligible].copy()
meta['TARGET'] = meta['event_date'].notna().astype('int8')

# A: chính xác hai snapshot đầu tiên. B: adopter trước first observed acquisition;
# non-adopter mọi snapshot còn quan sát được trong năm quan hệ đầu tiên.
# B có thể dùng snapshot SAU landmark: CHỈ LÀ CHẨN ĐOÁN, KHÔNG TRIỂN KHAI.
meta['B_cutoff'] = meta['event_date'].where(meta['TARGET'].eq(1), meta['end_observation'])
working = df.loc[df['ncodpers'].isin(meta.index)].copy()
working = working.join(meta[['landmark_date', 'B_cutoff']], on='ncodpers', validate='many_to_one')
A_rows = working.loc[working['snapshot_no'].le(2)].copy()
B_rows = working.loc[working['fecha_dato'] < working['B_cutoff']].copy()

def make_matrix(rows, name):
    rows = rows.sort_values(['ncodpers', 'fecha_dato'], kind='mergesort')
    last = rows.groupby('ncodpers', sort=False)[LAST_VALUE_FEATURES].last()
    first = rows.groupby('ncodpers', sort=False)[FIRST_VALUE_FEATURES].first()
    features = last.join(first)[MODEL_FEATURES]
    features.columns = [f'{name}_{c}' for c in features.columns]
    audit = rows.groupby('ncodpers', sort=False).agg(
        n_feature_snapshots=('fecha_dato', 'size'),
        last_feature_date=('fecha_dato', 'max'))
    audit.columns = [f'{name}_{c}' for c in audit.columns]
    return features.join(audit)

A = make_matrix(A_rows, 'A')
B = make_matrix(B_rows, 'B')
pair = meta[['TARGET', 'fecha_alta', 'landmark_date', 'event_date', 'B_cutoff']].join(A).join(B)
if pair.isna().all(axis=1).any() or A.index.difference(meta.index).size or B.index.difference(meta.index).size:
    raise AssertionError('Không khớp ID giữa cohort và đặc trưng.')
if len(pair) != len(meta) or pair[['A_n_feature_snapshots', 'B_n_feature_snapshots']].isna().any().any():
    raise AssertionError('A/B không có cùng tập khách hàng.')
assert (pair['A_n_feature_snapshots'] == 2).all()
assert (pair['A_last_feature_date'] == pair['landmark_date']).all()
assert (pair['B_last_feature_date'] < pair['B_cutoff']).all()
assert (pair['B_n_feature_snapshots'] >= pair['A_n_feature_snapshots']).all()

# Chia MỘT LẦN trên ID; dùng nguyên train/test cho hai nhánh.
ids_train, ids_test = train_test_split(
    pair.index.to_numpy(), test_size=0.20, stratify=pair['TARGET'].to_numpy(),
    random_state=SPLIT_SEED)
pair['split'] = 'test'
pair.loc[ids_train, 'split'] = 'train'
pair.index.name = 'ncodpers'

pair.to_csv(run_dir / 'Santander_A_B_paired_audit.csv', index=True)
for branch in ['A', 'B']:
    columns = ['TARGET', 'split'] + [f'{branch}_{c}' for c in MODEL_FEATURES]
    out = pair[columns].copy()
    out.columns = ['TARGET', 'split'] + MODEL_FEATURES
    out.to_csv(run_dir / f'Santander_Branch_{branch}_model_matrix.csv', index=True)

summary = {
    'source': str(INPUT_FILE), 'n_eligible': len(pair),
    'positives': int(pair['TARGET'].sum()),
    'train_n': int((pair['split'] == 'train').sum()),
    'test_n': int((pair['split'] == 'test').sum()),
    'A_window': 'first two monthly snapshots (inclusive of landmark snapshot)',
    'B_cutoff_positive': 'first observed card-ownership snapshot (exclusive)',
    'B_cutoff_negative': 'end of first relationship year (exclusive)',
    'B_uses_post_landmark_data': True,
    'duplicate_customer_dates': duplicates,
    'split_seed': SPLIT_SEED,
}
(run_dir / 'construction_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print('Hoàn tất A/B cùng cohort, nhãn, train/test. Nhánh B chưa được huấn luyện.')


Mounted at /content/drive
Đọc: /content/drive/MyDrive/Santander_CreditCard.csv
Lưu: /content/drive/MyDrive/Santander_B_same_population/run_20260927T030201774746Z
{
  "source": "/content/drive/MyDrive/Santander_CreditCard.csv",
  "n_eligible": 231727,
  "positives": 1287,
  "train_n": 185381,
  "test_n": 46346,
  "A_window": "first two monthly snapshots (inclusive of landmark snapshot)",
  "B_cutoff_positive": "first observed card-ownership snapshot (exclusive)",
  "B_cutoff_negative": "end of first relationship year (exclusive)",
  "B_uses_post_landmark_data": true,
  "duplicate_customer_dates": 0,
  "split_seed": 42
}
Hoàn tất A/B cùng cohort, nhãn, train/test. Nhánh B chưa được huấn luyện.


NHÁNH C

In [7]:
"""Tạo Santander Branch C từ lần chạy A/B cùng cohort.

Branch C = toàn bộ 20 biến của B + tenure_at_cutoff (ngày).
Đây là nhánh chẩn đoán, không dùng làm mô hình dự báo tại landmark.
"""

from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import json

import numpy as np
import pandas as pd


# Có thể gán Path('/content/drive/MyDrive/Santander_B_same_population/run_...')
# Nếu để None, mã chọn lần chạy A/B gần nhất có đủ ba file đầu vào.
AB_RUN_DIR = None
AB_OUTPUT_PARENT = Path('/content/drive/MyDrive/Santander_B_same_population')

AUDIT_NAME = 'Santander_A_B_paired_audit.csv'
A_NAME = 'Santander_Branch_A_model_matrix.csv'
B_NAME = 'Santander_Branch_B_model_matrix.csv'
REQUIRED = (AUDIT_NAME, A_NAME, B_NAME)

MODEL_FEATURES = [
    'ind_empleado', 'pais_residencia', 'sexo', 'age', 'ind_nuevo',
    'antiguedad', 'indrel', 'indrel_1mes', 'tiprel_1mes', 'indresi',
    'indext', 'conyuemp', 'indfall', 'tipodom', 'cod_prov', 'nomprov',
    'ind_actividad_cliente', 'renta', 'segmento', 'canal_entrada',
]


def select_ab_run(explicit_dir=None, output_parent=AB_OUTPUT_PARENT):
    if explicit_dir is not None:
        run = Path(explicit_dir)
    else:
        runs = sorted(
            (p for p in Path(output_parent).glob('run_*')
             if p.is_dir() and all((p / name).is_file() for name in REQUIRED)),
            reverse=True,
        )
        if not runs:
            raise FileNotFoundError(
                f'Không tìm thấy run A/B đủ file trong {output_parent}. '
                'Chạy notebook nhánh A/B trước, hoặc đặt AB_RUN_DIR cụ thể.'
            )
        run = runs[0]
    missing = [name for name in REQUIRED if not (run / name).is_file()]
    if missing:
        raise FileNotFoundError(f'Run {run} thiếu file: {missing}')
    return run


def read_indexed_csv(path):
    frame = pd.read_csv(path, dtype={'ncodpers': 'string'}, low_memory=False)
    if 'ncodpers' not in frame.columns:
        raise ValueError(f'{path.name}: thiếu cột ncodpers.')
    if frame['ncodpers'].isna().any() or frame['ncodpers'].duplicated().any():
        raise ValueError(f'{path.name}: ncodpers rỗng hoặc trùng.')
    return frame.set_index('ncodpers')


def file_sha256(path):
    h = sha256()
    with path.open('rb') as stream:
        for part in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(part)
    return h.hexdigest()


def construct_c(ab_run_dir, output_dir=None):
    ab_run_dir = Path(ab_run_dir)
    audit = read_indexed_csv(ab_run_dir / AUDIT_NAME)
    A = read_indexed_csv(ab_run_dir / A_NAME)
    B = read_indexed_csv(ab_run_dir / B_NAME)

    if len(audit) == 0 or set(audit.index) != set(A.index) or set(audit.index) != set(B.index):
        raise ValueError('A, B và audit phải có cùng tập ncodpers, không rỗng.')
    # Khớp đúng thứ tự hàng với nhánh B để thuận tiện so sánh điểm dự báo.
    audit = audit.reindex(B.index)
    A = A.reindex(B.index)
    if B.columns.tolist() != ['TARGET', 'split'] + MODEL_FEATURES:
        raise ValueError('Schema nhánh B khác mã A/B đã cung cấp; cần kiểm tra trước khi ghép C.')
    if A.columns.tolist() != B.columns.tolist():
        raise ValueError('Schema nhánh A và B không giống nhau.')
    for frame, name in ((A, 'A'), (B, 'B')):
        if not frame['TARGET'].equals(audit['TARGET']) or not frame['split'].equals(audit['split']):
            raise ValueError(f'Nhãn hoặc split ở {name} không khớp paired audit.')

    for column in ('fecha_alta', 'B_cutoff', 'B_last_feature_date', 'event_date'):
        if column not in audit.columns:
            raise ValueError(f'Paired audit thiếu {column}.')
    start = pd.to_datetime(audit['fecha_alta'], errors='raise')
    cutoff = pd.to_datetime(audit['B_cutoff'], errors='raise')
    last_feature_date = pd.to_datetime(audit['B_last_feature_date'], errors='raise')
    event = pd.to_datetime(audit['event_date'], errors='raise')
    if start.isna().any() or cutoff.isna().any() or last_feature_date.isna().any():
        raise ValueError('Ngày bắt đầu, cutoff hoặc ngày đặc trưng cuối bị thiếu.')
    if (last_feature_date >= cutoff).any():
        raise ValueError('Có đặc trưng B tại hoặc sau cutoff.')
    if not audit['TARGET'].isin([0, 1]).all() or not audit['split'].isin(['train', 'test']).all():
        raise ValueError('TARGET hoặc split không hợp lệ.')
    positives = audit['TARGET'].eq(1)
    if event[positives].isna().any() or event[~positives].notna().any():
        raise ValueError('Nhãn không khớp ngày phát hiện thẻ.')
    if not (cutoff[positives] == event[positives]).all():
        raise ValueError('Cutoff của adopter phải bằng ngày đầu tiên quan sát sở hữu thẻ.')

    tenure = (cutoff - start).dt.days
    if tenure.isna().any() or (tenure < 0).any():
        raise ValueError('tenure_at_cutoff thiếu hoặc âm; không tự sửa giá trị bất hợp lý.')
    C = B.copy()
    C['tenure_at_cutoff'] = tenure.astype('int64')
    pd.testing.assert_frame_equal(C[B.columns], B)
    if not np.array_equal(C.index.to_numpy(), B.index.to_numpy()):
        raise AssertionError('Thứ tự khách hàng của C đã thay đổi.')

    if output_dir is None:
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        output_dir = ab_run_dir / f'branch_C_{stamp}'
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)

    C.to_csv(output_dir / 'Santander_Branch_C_model_matrix.csv', index=True)
    c_audit = audit[[
        'TARGET', 'split', 'fecha_alta', 'landmark_date', 'event_date',
        'B_cutoff', 'B_last_feature_date', 'B_n_feature_snapshots',
    ]].copy()
    c_audit['tenure_at_cutoff'] = C['tenure_at_cutoff']
    c_audit.to_csv(output_dir / 'Santander_Branch_C_audit.csv', index=True)

    summary = {
        'branch': 'Santander C = paired B + tenure_at_cutoff',
        'definition': 'tenure_at_cutoff = (B_cutoff - fecha_alta).days',
        'unit': 'days', 'status': 'diagnostic_only_not_landmark_deployable',
        'ab_run_dir': str(ab_run_dir), 'output_dir': str(output_dir),
        'source_sha256': {name: file_sha256(ab_run_dir / name) for name in REQUIRED},
        'n_customers': int(len(C)),
        'n_positive': int(positives.sum()),
        'n_train': int(C['split'].eq('train').sum()),
        'n_test': int(C['split'].eq('test').sum()),
        'n_features_B': len(MODEL_FEATURES), 'n_features_C': len(MODEL_FEATURES) + 1,
        'tenure_days_min': int(tenure.min()),
        'tenure_days_median': float(tenure.median()),
        'tenure_days_max': int(tenure.max()),
    }
    (output_dir / 'Santander_Branch_C_summary.json').write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    return output_dir, summary


if __name__ == '__main__':
    try:
        from google.colab import drive
    except ImportError:
        pass
    else:
        drive.mount('/content/drive')
    selected = select_ab_run(AB_RUN_DIR)
    print('Nguồn A/B:', selected)
    created, result = construct_c(selected)
    print('Kết quả C:', created)
    print(json.dumps(result, ensure_ascii=False, indent=2))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Nguồn A/B: /content/drive/MyDrive/Santander_B_same_population/run_20260927T030201774746Z
Kết quả C: /content/drive/MyDrive/Santander_B_same_population/run_20260927T030201774746Z/branch_C_20260927T031233994816Z
{
  "branch": "Santander C = paired B + tenure_at_cutoff",
  "definition": "tenure_at_cutoff = (B_cutoff - fecha_alta).days",
  "unit": "days",
  "status": "diagnostic_only_not_landmark_deployable",
  "ab_run_dir": "/content/drive/MyDrive/Santander_B_same_population/run_20260927T030201774746Z",
  "output_dir": "/content/drive/MyDrive/Santander_B_same_population/run_20260927T030201774746Z/branch_C_20260927T031233994816Z",
  "source_sha256": {
    "Santander_A_B_paired_audit.csv": "bebc13d03deb9aa34dcdb0d0f88b60e962c77e195a5edc08627b8ace7b9d4ef5",
    "Santander_Branch_A_model_matrix.csv": "ab55f01dda8c67153a41f0c914d25bd4d914b423140a3c7ad011ff1908c297ad"

ABC

"""Chạy Santander A/B/C trên CHÍNH cohort, nhãn và split của A Fixed60D.

Đầu vào: Santander_CreditCard.csv + Fixed60D_Audit.csv + Fixed60D_Model.csv
từ notebook A mà tác giả đã gửi. B/C cũ dùng hai snapshot KHÔNG phải đầu vào.
Chạy trực tiếp trong Google Colab bằng notebook đi kèm.
"""

from datetime import datetime, timedelta, timezone
from pathlib import Path
from hashlib import sha256
import gc
import json
import platform
import shutil

import duckdb
import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost
from scipy.stats import norm, rankdata
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, matthews_corrcoef, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier


ROOT = Path('/content/drive/MyDrive')
RAW_FILE = ROOT / 'Santander_CreditCard.csv'
A_AUDIT_FILE = ROOT / 'Santander_CreditCard_Fixed60D_Audit.csv'
A_MODEL_FILE = ROOT / 'Santander_CreditCard_Fixed60D_Model.csv'
A_OLD_PREDICTIONS = (ROOT / 'Santander_External_Validation_Results' /
    'Fixed60D_XGBoost_NoSampling_SHAP' /
    'santander_fixed60d_nosampling_test_predictions.csv')
OUTPUT_PARENT = ROOT / 'Santander_ABC_Fixed60D_same_A_split'

FEATURES = [
    'IND_EMPLEADO', 'PAIS_RESIDENCIA', 'SEXO', 'AGE', 'IND_NUEVO',
    'ANTIGUEDAD', 'INDREL', 'INDREL_1MES', 'TIPREL_1MES', 'INDRESI',
    'INDEXT', 'CONYUEMP', 'INDFALL', 'TIPODOM', 'COD_PROV', 'NOMPROV',
    'IND_ACTIVIDAD_CLIENTE', 'RENTA', 'SEGMENTO', 'CANAL_ENTRADA',
]
XGB_PARAMS = dict(n_estimators=300, max_depth=6, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  objective='binary:logistic', eval_metric='logloss',
                  tree_method='hist', max_bin=256, random_state=42, n_jobs=2)
SPLIT_SEED = 42
CV_SEED = 42
N_FOLDS = 5
N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 20260927


def sha_file(path):
    h = sha256()
    with Path(path).open('rb') as f:
        for piece in iter(lambda: f.read(1024 * 1024), b''):
            h.update(piece)
    return h.hexdigest()


def load_a():
    for path in (RAW_FILE, A_AUDIT_FILE, A_MODEL_FILE):
        if not path.is_file():
            raise FileNotFoundError(f'Thiếu file đầu vào: {path}')
    audit = pd.read_csv(A_AUDIT_FILE, low_memory=False)
    model = pd.read_csv(A_MODEL_FILE, low_memory=False)
    audit.columns = audit.columns.str.strip().str.upper()
    model.columns = model.columns.str.strip().str.upper()
    needed = ['NCODPERS', 'TARGET_60D', 'FECHA_ALTA', 'CUTOFF_DATE'] + FEATURES
    if any(c not in audit for c in needed):
        raise ValueError(f'A audit thiếu: {sorted(set(needed) - set(audit.columns))}')
    if model.columns.tolist() != FEATURES + ['TARGET_60D']:
        raise ValueError('A model không đúng 20 biến + TARGET_60D theo file A đã gửi.')
    if len(model) != len(audit) or not len(model):
        raise ValueError('A model/audit không khớp số dòng hoặc rỗng.')
    # A model được ghi trực tiếp từ A audit sau khi bỏ ID/date: thứ tự dòng
    # chính là khóa ghép duy nhất vì A model không lưu ncodpers.
    same = model.eq(audit[model.columns]).fillna(False) | (
        model.isna() & audit[model.columns].isna())
    if not bool(same.to_numpy().all()):
        raise ValueError('A model và A audit lệch hàng hoặc nội dung. '
                         'Không thể ghép an toàn do A model không có ncodpers.')
    ids = pd.to_numeric(audit['NCODPERS'], errors='raise')
    if ids.isna().any() or (ids != np.floor(ids)).any():
        raise ValueError('ID trong A audit không phải số nguyên hợp lệ.')
    audit['NCODPERS'] = ids.astype('int64')
    if audit['NCODPERS'].duplicated().any():
        raise ValueError('ID khách hàng trong A audit trùng.')
    audit['TARGET_60D'] = pd.to_numeric(audit['TARGET_60D'], errors='raise').astype('int8')
    if not audit['TARGET_60D'].isin([0, 1]).all():
        raise ValueError('Nhãn A phải là 0/1.')

In [11]:
# SANTANDER A/B/C: CHẠY TOÀN BỘ TRONG MỘT Ô COLAB
# Cần ba file trong /content/drive/MyDrive: Santander_CreditCard.csv,
# Santander_CreditCard_Fixed60D_Audit.csv, Santander_CreditCard_Fixed60D_Model.csv.
# Nếu đã có kết quả, đổi RUN_MODE = 'train' thành 'view' ở phần cấu hình bên dưới.
# Dùng Runtime > Run all. Các kết quả sẽ hiện ra sau khi hoàn tất.
import importlib.util
import subprocess
import sys
for _package in ('duckdb', 'xgboost'):
    if importlib.util.find_spec(_package) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _package])

"""Chạy Santander A/B/C trên CHÍNH cohort, nhãn và split của A Fixed60D.

Đầu vào: Santander_CreditCard.csv + Fixed60D_Audit.csv + Fixed60D_Model.csv
từ notebook A mà tác giả đã gửi. B/C cũ dùng hai snapshot KHÔNG phải đầu vào.
Chạy trực tiếp trong Google Colab bằng notebook đi kèm.
"""

from datetime import datetime, timedelta, timezone
from pathlib import Path
from hashlib import sha256
import gc
import json
import platform
import shutil

import duckdb
import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost
from scipy.stats import norm, rankdata
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, matthews_corrcoef, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier


ROOT = Path('/content/drive/MyDrive')
RAW_FILE = ROOT / 'Santander_CreditCard.csv'
A_AUDIT_FILE = ROOT / 'Santander_CreditCard_Fixed60D_Audit.csv'
A_MODEL_FILE = ROOT / 'Santander_CreditCard_Fixed60D_Model.csv'
A_OLD_PREDICTIONS = (ROOT / 'Santander_External_Validation_Results' /
    'Fixed60D_XGBoost_NoSampling_SHAP' /
    'santander_fixed60d_nosampling_test_predictions.csv')
OUTPUT_PARENT = ROOT / 'Santander_ABC_Fixed60D_same_A_split'
# Đổi thành 'view' để xem lại CSV kết quả mà KHÔNG huấn luyện lại.
RUN_MODE = 'train'  # 'train' hoặc 'view'
VIEW_RUN_DIR = None  # Nếu xem một run cụ thể: Path('/content/drive/MyDrive/.../run_...')

FEATURES = [
    'IND_EMPLEADO', 'PAIS_RESIDENCIA', 'SEXO', 'AGE', 'IND_NUEVO',
    'ANTIGUEDAD', 'INDREL', 'INDREL_1MES', 'TIPREL_1MES', 'INDRESI',
    'INDEXT', 'CONYUEMP', 'INDFALL', 'TIPODOM', 'COD_PROV', 'NOMPROV',
    'IND_ACTIVIDAD_CLIENTE', 'RENTA', 'SEGMENTO', 'CANAL_ENTRADA',
]
XGB_PARAMS = dict(n_estimators=300, max_depth=6, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  objective='binary:logistic', eval_metric='logloss',
                  tree_method='hist', max_bin=256, random_state=42, n_jobs=2)
SPLIT_SEED = 42
CV_SEED = 42
N_FOLDS = 5
N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 20260927


def sha_file(path):
    h = sha256()
    with Path(path).open('rb') as f:
        for piece in iter(lambda: f.read(1024 * 1024), b''):
            h.update(piece)
    return h.hexdigest()


def load_a():
    for path in (RAW_FILE, A_AUDIT_FILE, A_MODEL_FILE):
        if not path.is_file():
            raise FileNotFoundError(f'Thiếu file đầu vào: {path}')
    audit = pd.read_csv(A_AUDIT_FILE, low_memory=False)
    model = pd.read_csv(A_MODEL_FILE, low_memory=False)
    audit.columns = audit.columns.str.strip().str.upper()
    model.columns = model.columns.str.strip().str.upper()
    needed = ['NCODPERS', 'TARGET_60D', 'FECHA_ALTA', 'CUTOFF_DATE'] + FEATURES
    if any(c not in audit for c in needed):
        raise ValueError(f'A audit thiếu: {sorted(set(needed) - set(audit.columns))}')
    if model.columns.tolist() != FEATURES + ['TARGET_60D']:
        raise ValueError('A model không đúng 20 biến + TARGET_60D theo file A đã gửi.')
    if len(model) != len(audit) or not len(model):
        raise ValueError('A model/audit không khớp số dòng hoặc rỗng.')
    # A model được ghi trực tiếp từ A audit sau khi bỏ ID/date: thứ tự dòng
    # chính là khóa ghép duy nhất vì A model không lưu ncodpers.
    same = model.eq(audit[model.columns]).fillna(False) | (
        model.isna() & audit[model.columns].isna())
    if not bool(same.to_numpy().all()):
        raise ValueError('A model và A audit lệch hàng hoặc nội dung. '
                         'Không thể ghép an toàn do A model không có ncodpers.')
    ids = pd.to_numeric(audit['NCODPERS'], errors='raise')
    if ids.isna().any() or (ids != np.floor(ids)).any():
        raise ValueError('ID trong A audit không phải số nguyên hợp lệ.')
    audit['NCODPERS'] = ids.astype('int64')
    if audit['NCODPERS'].duplicated().any():
        raise ValueError('ID khách hàng trong A audit trùng.')
    audit['TARGET_60D'] = pd.to_numeric(audit['TARGET_60D'], errors='raise').astype('int8')
    if not audit['TARGET_60D'].isin([0, 1]).all():
        raise ValueError('Nhãn A phải là 0/1.')
    audit['FECHA_ALTA'] = pd.to_datetime(audit['FECHA_ALTA'], errors='raise')
    audit['CUTOFF_DATE'] = pd.to_datetime(audit['CUTOFF_DATE'], errors='raise')
    if audit[['FECHA_ALTA', 'CUTOFF_DATE']].isna().any().any():
        raise ValueError('Thiếu ngày trong audit A.')
    if not (audit['CUTOFF_DATE'] - audit['FECHA_ALTA']).dt.days.eq(60).all():
        raise ValueError('A không sử dụng cutoff đúng 60 ngày.')
    audit['ROW_ID'] = np.arange(1, len(audit) + 1).astype(str)
    model['ROW_ID'] = audit['ROW_ID'].to_numpy()
    audit = audit.sort_values('ROW_ID').reset_index(drop=True)
    model = model.sort_values('ROW_ID').drop(columns='ROW_ID').reset_index(drop=True)
    # Đây là đúng cách code XGB No Sampling của A tạo train/test.
    tr, te = train_test_split(np.arange(len(audit)), test_size=0.20,
                             stratify=audit['TARGET_60D'], random_state=SPLIT_SEED)
    split = np.full(len(audit), 'train', dtype=object)
    split[te] = 'test'
    audit['SPLIT'] = split
    if A_OLD_PREDICTIONS.is_file():
        old = pd.read_csv(A_OLD_PREDICTIONS, dtype={'ROW_ID': 'string'})
        old.columns = old.columns.str.upper()
        if 'ROW_ID' not in old or 'Y_TRUE' not in old:
            raise ValueError('File predictions A cũ thiếu ROW_ID/Y_TRUE.')
        if (len(old) != len(te) or
            old['ROW_ID'].astype(str).tolist() != audit.iloc[te]['ROW_ID'].tolist() or
            not np.array_equal(old['Y_TRUE'].to_numpy(), audit.iloc[te]['TARGET_60D'].to_numpy())):
            raise ValueError('Train/test tái tạo KHÔNG khớp predictions A cũ; '
                             'kiểm tra phiên bản A audit/model trước khi chạy.')
        print('Đã xác nhận ID và nhãn test với dự báo A cũ:', A_OLD_PREDICTIONS)
        old_split_verified = True
    else:
        print('Không có predictions A cũ; split tái tạo từ đúng thuật toán A '
              '(ROW_ID chuỗi, sort, stratify, seed=42).')
        old_split_verified = False
    return audit, model, tr, te, old_split_verified


def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


def normalized_feature(col):
    raw = f'TRIM(s."{col.lower()}")'
    return (f'CASE WHEN UPPER({raw}) IN '
            "('', 'NA', 'N/A', 'NAN', 'NULL', 'NONE') "
            f'THEN NULL ELSE {raw} END AS "{col}"')


def build_b(audit, work_dir):
    con = duckdb.connect(str(work_dir / 'selected.duckdb'))
    con.execute("SET memory_limit='4GB'")
    con.execute('SET threads=2')
    con.execute(f'SET temp_directory={sql_string(work_dir / "spill")}')
    source = (f'read_csv({sql_string(RAW_FILE)}, header=true, '
              'all_varchar=true, sample_size=10000)')
    c = audit[['NCODPERS', 'FECHA_ALTA', 'CUTOFF_DATE']].copy()
    c.columns = ['ncodpers', 'fecha_alta', 'cutoff_date']
    con.register('cohort_a', c)
    try:
        end_raw = con.execute(f'SELECT MAX(TRY_CAST(fecha_dato AS DATE)) FROM {source}').fetchone()[0]
        if end_raw is None:
            raise ValueError('Không có snapshot ngày hợp lệ trong nguồn Santander.')
        admin_end = pd.Timestamp(end_raw) + pd.Timedelta(days=1)
        fields = ',\n            '.join(normalized_feature(col) for col in FEATURES)
        con.execute(f'''
          CREATE TABLE selected AS
          SELECT c.ncodpers,
                 TRY_CAST(s.fecha_dato AS DATE) AS fecha_dato,
                 CASE WHEN TRY_CAST(s.ind_tjcr_fin_ult1 AS DOUBLE)>0 THEN 1 ELSE 0 END AS has_card,
                 {fields}
          FROM {source} AS s
          INNER JOIN cohort_a AS c
            ON TRY_CAST(s.ncodpers AS BIGINT) = c.ncodpers
          WHERE TRY_CAST(s.fecha_dato AS DATE) >= c.fecha_alta
        ''')
        dup = con.execute('''SELECT ncodpers, fecha_dato, COUNT(*) AS n
            FROM selected GROUP BY 1, 2 HAVING COUNT(*)>1 LIMIT 1''').fetchone()
        if dup:
            raise ValueError(f'Nguồn có snapshot trùng ID/ngày {dup}; '
                             'phải xác lập cách xử lý trước khi so sánh A/B/C.')
        event = con.execute('''
           SELECT s.ncodpers,
                  SUM(CASE WHEN s.has_card=1 AND s.fecha_dato<=c.cutoff_date
                      THEN 1 ELSE 0 END) AS early_card_rows,
                  MIN(s.fecha_dato) FILTER (WHERE s.has_card=1
                      AND s.fecha_dato>c.cutoff_date) AS event_date,
                  COUNT(*) FILTER (WHERE s.fecha_dato>c.cutoff_date) AS n_followup_snapshots,
                  COUNT(*) AS n_source_snapshots
           FROM selected AS s JOIN cohort_a AS c USING(ncodpers)
           GROUP BY s.ncodpers
        ''').fetchdf().set_index('ncodpers')
        if set(event.index) != set(audit['NCODPERS']):
            raise ValueError('Nguồn gốc thiếu khách hàng A hoặc ID không khớp.')
        event = event.reindex(audit['NCODPERS'].to_numpy())
        if event['early_card_rows'].sum() != 0:
            raise ValueError('Nguồn có thẻ trong 60 ngày nhưng A audit vẫn giữ khách hàng.')
        events = pd.to_datetime(event['event_date'])
        if not np.array_equal(events.notna().to_numpy(), audit['TARGET_60D'].to_numpy().astype(bool)):
            raise ValueError('Nhãn A không khớp event từ CSV gốc; có thể file nguồn đã đổi.')
        no_followup = int(event['n_followup_snapshots'].eq(0).sum())
        if no_followup:
            print(f'Lưu ý: A có {no_followup:,} khách hàng không có snapshot sau ngày 60; '
                  'giữ nhãn A=0 để so sánh cùng cohort, cần nêu hạn chế này trong bài.')
        # Có ngày event trong A audit thì đối chiếu thêm, tránh nguồn bị thay.
        if 'FIRST_CARD_DATE_AFTER_CUTOFF' in audit:
            old_dates = pd.to_datetime(audit['FIRST_CARD_DATE_AFTER_CUTOFF'])
            old_array = old_dates.fillna(pd.Timestamp('1900-01-01')).to_numpy()
            new_array = events.fillna(pd.Timestamp('1900-01-01')).to_numpy()
            if not np.array_equal(old_array, new_array):
                raise ValueError('Ngày mở thẻ trong A audit lệch so với dữ liệu gốc.')
        cutoffs = audit[['NCODPERS']].rename(columns={'NCODPERS': 'ncodpers'}).copy()
        cutoffs['b_cutoff'] = events.reset_index(drop=True).fillna(admin_end)
        con.register('cutoffs_b', cutoffs)
        last_cols = FEATURES[:-1]
        expr = ',\n              '.join(
            f'ARG_MAX(s."{col}", s.fecha_dato) AS "{col}"' for col in last_cols)
        expr += ',\n              ARG_MIN(s."CANAL_ENTRADA", s.fecha_dato) AS "CANAL_ENTRADA"'
        b = con.execute(f'''
          SELECT s.ncodpers AS "NCODPERS",
                 COUNT(*) AS "B_N_FEATURE_SNAPSHOTS",
                 MAX(s.fecha_dato) AS "B_LAST_FEATURE_DATE",
                 {expr}
          FROM selected AS s JOIN cutoffs_b AS c USING(ncodpers)
          WHERE s.fecha_dato < c.b_cutoff
          GROUP BY s.ncodpers
        ''').fetchdf().set_index('NCODPERS')
        if set(b.index) != set(audit['NCODPERS']):
            raise ValueError('B mất khách hàng A; không được đổi cohort âm thầm.')
        b = b.reindex(audit['NCODPERS'].to_numpy())
        b['B_CUTOFF'] = cutoffs['b_cutoff'].to_numpy()
        b['EVENT_DATE'] = events.to_numpy()
        b['N_FOLLOWUP_SNAPSHOTS'] = event['n_followup_snapshots'].to_numpy()
        if not (pd.to_datetime(b['B_LAST_FEATURE_DATE']) < b['B_CUTOFF']).all():
            raise ValueError('Có snapshot B tại hoặc sau cutoff.')
        if 'N_FEATURE_SNAPSHOTS' in audit:
            if not (b['B_N_FEATURE_SNAPSHOTS'].to_numpy() >=
                    pd.to_numeric(audit['N_FEATURE_SNAPSHOTS']).to_numpy()).all():
                raise ValueError('Số snapshot B ít hơn A; nguồn/cửa sổ đã lệch.')
        return b.reset_index(drop=True), admin_end
    finally:
        con.close()


def infer_types(x):
    x = x.copy().replace(['', ' ', 'NA', 'N/A', 'na', 'null', 'NULL', 'None', 'none'], np.nan)
    for column in x.columns:
        if pd.api.types.is_object_dtype(x[column]) or pd.api.types.is_string_dtype(x[column]):
            converted = pd.to_numeric(x[column], errors='coerce')
            denom = int(x[column].notna().sum())
            if denom and converted.notna().sum() / denom >= 0.80:
                x[column] = converted
            else:
                x[column] = x[column].astype('string')
    cats = x.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    return x, cats


def preprocess(x_train, x_valid, cats):
    a, b = x_train.copy(), x_valid.copy()
    if cats:
        for col in cats:
            a[col] = a[col].astype('string').fillna('UNKNOWN')
            b[col] = b[col].astype('string').fillna('UNKNOWN')
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1,
                             encoded_missing_value=-1, dtype=np.float32)
        a[cats] = enc.fit_transform(a[cats])
        b[cats] = enc.transform(b[cats])
    for col in a:
        a[col] = pd.to_numeric(a[col], errors='coerce')
        b[col] = pd.to_numeric(b[col], errors='coerce')
    imp = SimpleImputer(strategy='median')
    return imp.fit_transform(a).astype(np.float32), imp.transform(b).astype(np.float32)


def metrics(y, scores):
    y = np.asarray(y, dtype=int)
    scores = np.asarray(scores, dtype=float)
    predicted = (scores >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()
    fpr, tpr, _ = roc_curve(y, scores)
    brier = brier_score_loss(y, scores)
    prevalence = y.mean()
    return dict(AUC=roc_auc_score(y, scores), KS=float(np.max(tpr-fpr)),
                Recall=recall_score(y, predicted, zero_division=0),
                Precision=precision_score(y, predicted, zero_division=0),
                F1=f1_score(y, predicted, zero_division=0),
                Accuracy=accuracy_score(y, predicted), Brier=brier,
                BSS=1-brier/(prevalence*(1-prevalence)),
                Specificity=tn/(tn+fp), NPV=tn/(tn+fn) if (tn+fn) else np.nan,
                MCC=matthews_corrcoef(y, predicted),
                TP=int(tp), FP=int(fp), TN=int(tn), FN=int(fn))


def ranking(y, scores, branch):
    sorted_df = pd.DataFrame({'y': np.asarray(y), 'score': np.asarray(scores)})
    sorted_df = sorted_df.sort_values('score', ascending=False).reset_index(drop=True)
    n_pos, prevalence = int(sorted_df.y.sum()), float(sorted_df.y.mean())
    rows = []
    for pct in (10, 20, 30):
        n = max(1, int(np.ceil(len(y)*pct/100)))
        hits = int(sorted_df.y.iloc[:n].sum())
        precision = hits/n
        rows.append(dict(branch=branch, top_pct=pct, n_contact=n, hits=hits,
                         precision=precision, lift=precision/prevalence,
                         capture=hits/n_pos))
    return rows


def fit_branch(x, y, tr, te, branch):
    x, cats = infer_types(x)
    x_train, x_test = x.iloc[tr].reset_index(drop=True), x.iloc[te].reset_index(drop=True)
    y_train, y_test = y[tr], y[te]
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=CV_SEED)
    cv_rows = []
    for fold, (i, j) in enumerate(cv.split(x_train, y_train), 1):
        xt, xv = preprocess(x_train.iloc[i], x_train.iloc[j], cats)
        model = XGBClassifier(**XGB_PARAMS)
        model.fit(xt, y_train[i])
        p = model.predict_proba(xv)[:, 1]
        cv_rows.append(dict(branch=branch, fold=fold, **metrics(y_train[j], p)))
        print(f'{branch} CV {fold}/{N_FOLDS}: AUC={cv_rows[-1]["AUC"]:.6f}')
        del model, xt, xv
        gc.collect()
    xt, xv = preprocess(x_train, x_test, cats)
    model = XGBClassifier(**XGB_PARAMS)
    model.fit(xt, y_train)
    scores = model.predict_proba(xv)[:, 1].astype(float)
    result = dict(branch=branch, n_train=len(tr), n_test=len(te),
                  positives_train=int(y_train.sum()), positives_test=int(y_test.sum()),
                  **metrics(y_test, scores))
    print(f'{branch} TEST: AUC={result["AUC"]:.6f} | KS={result["KS"]:.6f} '
          f'| Brier={result["Brier"]:.6f}')
    return result, cv_rows, ranking(y_test, scores, branch), scores


def delong_pair(y, p, q):
    y = np.asarray(y).astype(bool)
    m, n = int(y.sum()), int((~y).sum())
    s = np.vstack([np.asarray(p), np.asarray(q)])
    positive, negative = s[:, y], s[:, ~y]
    tx = np.vstack([rankdata(row, method='average') for row in positive])
    ty = np.vstack([rankdata(row, method='average') for row in negative])
    tz = np.vstack([rankdata(row, method='average') for row in
                    np.concatenate([positive, negative], axis=1)])
    auc = (tz[:, :m].sum(axis=1)/m - (m+1)/2)/n
    v01 = (tz[:, :m] - tx)/n
    v10 = 1 - (tz[:, m:] - ty)/m
    cov = np.cov(v01)/m + np.cov(v10)/n
    variance = max(0.0, float(cov[0, 0]+cov[1, 1]-2*cov[0, 1]))
    diff = float(auc[0]-auc[1])
    if variance == 0:
        return 0.0 if diff else 1.0
    return float(2*norm.sf(abs(diff)/np.sqrt(variance)))


def contrasts(y, predictions):
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    pairs = [('B', 'A'), ('C', 'B'), ('C', 'A')]
    draws = {f'{a}-{b}': [] for a, b in pairs}
    n = len(y)
    for i in range(N_BOOTSTRAP):
        idx = rng.integers(0, n, size=n)
        sampled_y = y[idx]
        if sampled_y.min() == sampled_y.max():
            continue
        aucs = {k: roc_auc_score(sampled_y, v[idx]) for k, v in predictions.items()}
        for a, b in pairs:
            draws[f'{a}-{b}'].append(aucs[a]-aucs[b])
        if (i+1) % 200 == 0:
            print(f'Bootstrap {i+1}/{N_BOOTSTRAP}')
    if any(len(v) < 0.95*N_BOOTSTRAP for v in draws.values()):
        raise ValueError('Quá nhiều bootstrap chỉ có một lớp.')
    rows = []
    for a, b in pairs:
        vals = draws[f'{a}-{b}']
        rows.append(dict(contrast=f'{a}-{b}', auc_diff=roc_auc_score(y, predictions[a])-
                         roc_auc_score(y, predictions[b]),
                         ci_lower=float(np.quantile(vals, 0.025)),
                         ci_upper=float(np.quantile(vals, 0.975)),
                         delong_p=delong_pair(y, predictions[a], predictions[b]),
                         bootstrap_valid=len(vals)))
    return rows, pd.DataFrame(draws)


def show_saved_results(run_dir):
    """Đọc các file của CHÍNH run đã lưu và hiển thị trong Colab."""
    run_dir = Path(run_dir)
    tables = [
        ('BẢNG A/B/C NHƯ TABLE 8 VIB', 'table_like_VIB_Table8.csv', None),
        ('CHỈ SỐ HOLDOUT ĐẦY ĐỦ', 'holdout_metrics.csv',
         ['branch','n_train','n_test','positives_train','positives_test',
          'AUC','KS','Recall','Precision','F1','Accuracy','Brier','BSS',
          'Specificity','NPV','MCC','TP','FP','TN','FN']),
        ('5-FOLD CV (TRUNG BÌNH VÀ ĐỘ LỆCH CHUẨN)', 'cv_summary.csv', None),
        ('TOP 10/20/30%: PRECISION, LIFT, CAPTURE', 'ranking_top10_20_30.csv', None),
        ('SO SÁNH AUC GHÉP CẶP', 'paired_auc_contrasts.csv', None),
    ]
    try:
        from IPython.display import display
    except ImportError:
        display = lambda frame: print(frame.to_string(index=False))
    print('\nTHƯ MỤC KẾT QUẢ:', run_dir)
    for title, filename, columns in tables:
        file_path = run_dir / filename
        if not file_path.is_file():
            raise FileNotFoundError(f'Run chưa hoàn tất hoặc thiếu {file_path}')
        frame = pd.read_csv(file_path)
        if columns:
            frame = frame[columns]
        print('\n' + '='*18 + ' ' + title + ' ' + '='*18)
        display(frame.round(6))
    comparison = run_dir / 'A_original_vs_rerun.csv'
    if comparison.is_file():
        print('\nĐỐI CHIẾU A CŨ VÀ A CHẠY LẠI:')
        display(pd.read_csv(comparison).round(6))
    pred_file = run_dir / 'paired_holdout_predictions.csv'
    predictions = pd.read_csv(pred_file)
    print(f'\nDự báo từng khách hàng: {pred_file} ({len(predictions):,} dòng)')
    display(predictions.head(5).round(6))
    print('\nCác file CSV/JSON trong run:')
    for path in sorted(run_dir.iterdir()):
        if path.is_file():
            print(' •', path.name)


def main():
    try:
        from google.colab import drive
    except ImportError:
        pass
    else:
        if not ROOT.is_dir():
            drive.mount('/content/drive')
    if RUN_MODE == 'view':
        if VIEW_RUN_DIR is not None:
            selected = Path(VIEW_RUN_DIR)
        else:
            candidates = sorted(OUTPUT_PARENT.glob('run_*/holdout_metrics.csv'), reverse=True)
            if not candidates:
                raise FileNotFoundError(f'Chưa có kết quả trong {OUTPUT_PARENT}. '
                                        'Chạy RUN_MODE="train" trước.')
            selected = candidates[0].parent
        show_saved_results(selected)
        return
    if RUN_MODE != 'train':
        raise ValueError('RUN_MODE chỉ nhận "train" hoặc "view".')
    audit, a_model, tr, te, verified = load_a()
    print('A cohort:', len(audit), '| positives:', int(audit.TARGET_60D.sum()),
          '| train:', len(tr), '| test:', len(te))
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    out = OUTPUT_PARENT / f'run_{stamp}'
    out.mkdir(parents=True, exist_ok=False)
    work = Path('/content') / f'santander_abc_work_{stamp}'
    work.mkdir(parents=True, exist_ok=False)
    try:
        b, admin_end = build_b(audit, work)
        tenure = (b['B_CUTOFF'] - audit['FECHA_ALTA']).dt.days
        if tenure.isna().any() or (tenure < 0).any():
            raise ValueError('Tenure B/C không hợp lệ.')
        branch_x = {
            'A': a_model[FEATURES].copy(),
            'B': b[FEATURES].copy(),
            'C': b[FEATURES].copy(),
        }
        branch_x['C']['TENURE_AT_CUTOFF'] = tenure.astype('int64').to_numpy()
        if not branch_x['C'][FEATURES].equals(branch_x['B'][FEATURES]):
            raise AssertionError('Nhánh C đã vô tình đổi biến của B.')
        keys = audit[['NCODPERS', 'ROW_ID', 'TARGET_60D', 'SPLIT']]
        keys.to_csv(out / 'shared_A_split.csv', index=False)
        b_audit = keys.copy()
        b_audit['A_CUTOFF'] = audit['CUTOFF_DATE']
        b_audit['B_CUTOFF'] = b['B_CUTOFF'].to_numpy()
        b_audit['EVENT_DATE'] = b['EVENT_DATE'].to_numpy()
        b_audit['B_LAST_FEATURE_DATE'] = b['B_LAST_FEATURE_DATE'].to_numpy()
        b_audit['B_N_FEATURE_SNAPSHOTS'] = b['B_N_FEATURE_SNAPSHOTS'].to_numpy()
        b_audit['N_FOLLOWUP_SNAPSHOTS'] = b['N_FOLLOWUP_SNAPSHOTS'].to_numpy()
        b_audit['TENURE_AT_CUTOFF'] = tenure.to_numpy()
        b_audit.to_csv(out / 'paired_cutoff_audit.csv', index=False)
        y = audit['TARGET_60D'].to_numpy(dtype=np.int8)
        test_y = y[te]
        results, cvs, rankings, predictions = [], [], [], {}
        for name in ('A', 'B', 'C'):
            matrix = pd.concat([keys.reset_index(drop=True),
                                branch_x[name].reset_index(drop=True)], axis=1)
            matrix.to_csv(out / f'branch_{name}_matrix.csv', index=False)
            result, cv_rows, rank_rows, scores = fit_branch(branch_x[name], y, tr, te, name)
            results.append(result)
            cvs.extend(cv_rows)
            rankings.extend(rank_rows)
            predictions[name] = scores
        if verified:
            old = pd.read_csv(A_OLD_PREDICTIONS)
            old.columns = old.columns.str.upper()
            if 'Y_PROB' in old:
                old_auc = roc_auc_score(test_y, old['Y_PROB'].to_numpy())
                new_auc = next(r['AUC'] for r in results if r['branch']=='A')
                pd.DataFrame([dict(old_A_AUC=old_auc, rerun_A_AUC=new_auc,
                                   difference=new_auc-old_auc)]).to_csv(
                    out / 'A_original_vs_rerun.csv', index=False)
                print(f'A cũ AUC={old_auc:.6f}; A chạy lại AUC={new_auc:.6f}')
                if abs(old_auc-new_auc)>0.005:
                    print('AUC A thay đổi rõ: rà soát môi trường/thư viện trước khi diễn giải B/C.')
        all_predictions = audit.iloc[te][['NCODPERS', 'ROW_ID', 'TARGET_60D']].copy()
        for name in ('A', 'B', 'C'):
            all_predictions[f'SCORE_{name}'] = predictions[name]
        all_predictions.to_csv(out / 'paired_holdout_predictions.csv', index=False)
        pd.DataFrame(results).to_csv(out / 'holdout_metrics.csv', index=False)
        cvdf = pd.DataFrame(cvs)
        cvdf.to_csv(out / 'cv_folds.csv', index=False)
        summary_cols = ['AUC','KS','Recall','Precision','F1','Accuracy','Brier','BSS']
        cv_summary = cvdf.groupby('branch')[summary_cols].agg(['mean','std'])
        cv_summary.columns = [f'{a}_{b}' for a,b in cv_summary.columns]
        cv_summary.reset_index().to_csv(out / 'cv_summary.csv', index=False)
        pd.DataFrame(rankings).to_csv(out / 'ranking_top10_20_30.csv', index=False)
        r10 = {r['branch']: r for r in rankings if r['top_pct']==10}
        paper_rows = []
        for r in results:
            name = r['branch']
            paper_rows.append(dict(branch=name, AUC=r['AUC'],
                                   delta_AUC=None if name=='A' else
                                   r['AUC']-results[0 if name=='B' else 1]['AUC'],
                                   KS=r['KS'], BSS=r['BSS'],
                                   Precision_at_10=r10[name]['precision'],
                                   Lift_at_10=r10[name]['lift'],
                                   Capture_at_10=r10[name]['capture']))
        pd.DataFrame(paper_rows).to_csv(out / 'table_like_VIB_Table8.csv', index=False)
        delta_rows, bootstrap = contrasts(test_y, predictions)
        pd.DataFrame(delta_rows).to_csv(out / 'paired_auc_contrasts.csv', index=False)
        bootstrap.to_csv(out / 'paired_bootstrap_draws.csv', index=False)
        meta = dict(source_raw=str(RAW_FILE), a_audit=str(A_AUDIT_FILE),
                    a_model=str(A_MODEL_FILE), original_a_test_ids_verified=verified,
                    a_old_predictions=str(A_OLD_PREDICTIONS) if verified else None,
                    hashes={'a_audit':sha_file(A_AUDIT_FILE), 'a_model':sha_file(A_MODEL_FILE)},
                    n_customers=len(audit), positives=int(y.sum()),
                    train=len(tr), test=len(te), train_seed=SPLIT_SEED,
                    n_without_post60_snapshot=int(b['N_FOLLOWUP_SNAPSHOTS'].eq(0).sum()),
                    cv_seed=CV_SEED, bootstrap_seed=BOOTSTRAP_SEED,
                    bootstrap_n=N_BOOTSTRAP, administrative_end_exclusive=str(admin_end.date()),
                    protocol='A 60-day inclusive; B first observed card after day 60 exclusive, '
                             'nonadopter common end exclusive; C=B+tenure days',
                    model='XGBoost no sampling; separately fitted A/B/C',
                    environment={'python':platform.python_version(), 'numpy':np.__version__,
                                 'pandas':pd.__version__, 'scikit_learn':sklearn.__version__,
                                 'xgboost':xgboost.__version__, 'duckdb':duckdb.__version__,
                                 'scipy':scipy.__version__})
        (out / 'run_metadata.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding='utf-8')
        print('\nKẾT QUẢ HOLDOUT (cùng ID/nhãn/split):')
        print(pd.DataFrame(results)[['branch','AUC','KS','Recall','Precision','F1',
                                     'Accuracy','Brier','BSS']].to_string(index=False))
        print('\nHIỆU AUC GHÉP CẶP:')
        print(pd.DataFrame(delta_rows).to_string(index=False))
        print('\nĐã lưu:', out)
        show_saved_results(out)
    finally:
        shutil.rmtree(work, ignore_errors=True)


if __name__ == '__main__':
    main()


Đã xác nhận ID và nhãn test với dự báo A cũ: /content/drive/MyDrive/Santander_External_Validation_Results/Fixed60D_XGBoost_NoSampling_SHAP/santander_fixed60d_nosampling_test_predictions.csv
A cohort: 153399 | positives: 992 | train: 122719 | test: 30680


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Lưu ý: A có 14,332 khách hàng không có snapshot sau ngày 60; giữ nhãn A=0 để so sánh cùng cohort, cần nêu hạn chế này trong bài.
A CV 1/5: AUC=0.886144
A CV 2/5: AUC=0.866673
A CV 3/5: AUC=0.867376
A CV 4/5: AUC=0.878026
A CV 5/5: AUC=0.873263
A TEST: AUC=0.886585 | KS=0.641884 | Brier=0.006331
B CV 1/5: AUC=0.968343
B CV 2/5: AUC=0.955700
B CV 3/5: AUC=0.960151
B CV 4/5: AUC=0.962694
B CV 5/5: AUC=0.961398
B TEST: AUC=0.960336 | KS=0.765124 | Brier=0.003671
C CV 1/5: AUC=0.992803
C CV 2/5: AUC=0.986717
C CV 3/5: AUC=0.992857
C CV 4/5: AUC=0.993771
C CV 5/5: AUC=0.990872
C TEST: AUC=0.990719 | KS=0.909539 | Brier=0.002150
A cũ AUC=0.886585; A chạy lại AUC=0.886585
Bootstrap 200/1000
Bootstrap 400/1000
Bootstrap 600/1000
Bootstrap 800/1000
Bootstrap 1000/1000

KẾT QUẢ HOLDOUT (cùng ID/nhãn/split):
branch      AUC       KS   Recall  Precision       F1  Accuracy    Brier      BSS
     A 0.886585 0.641884 0.000000   0.000000 0.000000  0.993546 0.006331 0.012690
     B 0.960336 0.765124 0.3

,branch,AUC,delta_AUC,KS,BSS,Precision_at_10,Lift_at_10,Capture_at_10
0,A,0.886585,NaN,0.641884,0.012690,0.037484,5.808081,0.580808
1,B,0.960336,0.073750,0.765124,0.427486,0.055085,8.535354,0.853535
2,C,0.990719,0.030383,0.909539,0.664726,0.062581,9.696970,0.969697


...


ĐỐI CHIẾU A CŨ VÀ A CHẠY LẠI:


,old_A_AUC,rerun_A_AUC,difference
0,0.886585,0.886585,0.0



Dự báo từng khách hàng: /content/drive/MyDrive/Santander_ABC_Fixed60D_same_A_split/run_20260927T034225723501Z/paired_holdout_predictions.csv (30,680 dòng)


,NCODPERS,ROW_ID,TARGET_60D,SCORE_A,SCORE_B,SCORE_C
0,1453818,66301,0,0.000075,0.000008,0.000008
1,1543525,145956,0,0.000060,0.000014,0.000005
2,1399413,20184,0,0.015787,0.042849,0.042941
3,1530077,135239,0,0.006480,0.008655,0.005866
4,1552461,152856,0,0.000090,0.000042,0.000015



Các file CSV/JSON trong run:
 • A_original_vs_rerun.csv
 • branch_A_matrix.csv
 • branch_B_matrix.csv
 • branch_C_matrix.csv
 • cv_folds.csv
 • cv_summary.csv
 • holdout_metrics.csv
 • paired_auc_contrasts.csv
 • paired_bootstrap_draws.csv
 • paired_cutoff_audit.csv
 • paired_holdout_predictions.csv
 • ranking_top10_20_30.csv
 • run_metadata.json
 • shared_A_split.csv
 • table_like_VIB_Table8.csv
